<a href="https://colab.research.google.com/github/Novation257/cs325-project/blob/main/src/qwasr_combiner_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QwASR Data Combiner - combines the QwASR and Yahoo Finance data and runs preprocessing to create the final dataset

In [283]:
# Imports
from bs4 import BeautifulSoup
import pandas as pd
import time
import dateutil
import warnings
import yfinance as yf
from datetime import datetime
from datetime import timedelta
import numpy as np
import random

# Supress certain warnings
from dateutil.parser import UnknownTimezoneWarning
warnings.filterwarnings("ignore", category=UnknownTimezoneWarning)

In [284]:
# Open CSV containing forbes and yf data
fyf_df = pd.DataFrame(pd.read_csv("forbes_and_yahoo_738.csv"))
dates = fyf_df['Time'].tolist()
numDates = len(dates)

fyf_df.head()

,Unnamed: 0,Link,Title,Time,Author,Body,T-3D,T-1D,T-0D,T+1D,T+3D
0,0,https://www.forbes.com/sites/catherinebrock/20...,Stock Prices Finish Week Lower After Put Filin...,"Nov 07, 2025, 06:30pm EST",Catherine Brock,U.S. stocks retreated this week after investor...,198.668182,188.059357,188.129349,199.028152,193.778732
1,1,https://www.forbes.com/sites/greatspeculations...,A Decade Of Rewards: $83 Bil From NVIDIA Stock,"Nov 07, 2025, 11:55am EST",Trefis Team,"Over the past ten years, NVIDIA (NVDA) stock h...",198.668182,188.059357,188.129349,199.028152,193.778732
2,2,https://www.forbes.com/sites/johnkoetsier/2025...,8 Robotics Startups Backed By Nvidia And Amazon,"Nov 06, 2025, 02:20pm EST",John Koetsier,Amazon and Nvidia are backing eight AI and rob...,206.857285,195.188583,188.059357,188.129349,193.138794
3,3,https://www.forbes.com/sites/ywang/2025/11/05/...,Founder Of The ‘Nvidia Of China’ Triples His W...,"Nov 05, 2025, 04:39pm EST",Yue Wang,This story is part of Forbes’ coverage of Chin...,202.467773,198.668182,195.188583,188.059357,199.028152
4,4,https://www.forbes.com/sites/jonmarkman/2025/1...,You Missed Nvidia Because You Think In Straigh...,"Nov 04, 2025, 10:57am EST",Jon Markman,Your greatest risk in this market isn’t what y...,202.867722,206.857285,198.668182,195.188583,188.129349


In [285]:
# Open CSV containing QwASR predictions
q_df = pd.DataFrame(pd.read_csv("qwasr_scores.csv"))
q_df.head()

,relevancy,sentiment,qwasr_T+1D,qwasr_T+3D
0,0.90,-0.5,188.13,188.13
1,0.70,0.8,203.48,204.49
2,0.95,0.5,160.00,161.00
3,1.00,1.0,158.50,160.00
4,0.70,-0.5,155.50,158.00


In [286]:
# Create a new combined dataframe
df_combined = pd.concat([fyf_df, q_df], axis=1)
df_combined.head(2)

,Unnamed: 0,Link,Title,Time,Author,Body,T-3D,T-1D,T-0D,T+1D,T+3D,relevancy,sentiment,qwasr_T+1D,qwasr_T+3D
0,0,https://www.forbes.com/sites/catherinebrock/20...,Stock Prices Finish Week Lower After Put Filin...,"Nov 07, 2025, 06:30pm EST",Catherine Brock,U.S. stocks retreated this week after investor...,198.668182,188.059357,188.129349,199.028152,193.778732,0.9,-0.5,188.13,188.13
1,1,https://www.forbes.com/sites/greatspeculations...,A Decade Of Rewards: $83 Bil From NVIDIA Stock,"Nov 07, 2025, 11:55am EST",Trefis Team,"Over the past ten years, NVIDIA (NVDA) stock h...",198.668182,188.059357,188.129349,199.028152,193.778732,0.7,0.8,203.48,204.49


In [287]:
# Extra EDA - print min and max of each column, with index and article title
for col_name in df_combined.drop(columns=['Link', 'Title', 'Time', 'Author', 'Body']):
  print(col_name, ":")
  print("  min:", df_combined[col_name].min(), df_combined[col_name].idxmin())
  print("      ", df_combined.at[df_combined[col_name].idxmin(), 'Title'])
  print("  max:", df_combined[col_name].max(), df_combined[col_name].idxmax())
  print("     ", df_combined.at[df_combined[col_name].idxmax(), 'Title'])

Unnamed: 0 :
  min: 0 0
       Stock Prices Finish Week Lower After Put Filing On Nvidia, Palantir
  max: 737 737
      Nvidia CFO White Exits For "Personal Reasons"
T-3D :
  min: 0.2890163958072662 732
       Cray Claims The Fastest Supercomputing Crown With The NVIDIA-Powered Titan
  max: 207.01727294921875 5
      All Roads Lead To NVIDIA: Bankrolling Its Own AI Gold Rush
T-1D :
  min: 0.2793901264667511 732
       Cray Claims The Fastest Supercomputing Crown With The NVIDIA-Powered Titan
  max: 207.01727294921875 12
      Forbes Daily: AI Mints The Most Valuable Company Ever—Again
T-0D :
  min: 0.2732018828392029 732
       Cray Claims The Fastest Supercomputing Crown With The NVIDIA-Powered Titan
  max: 207.01727294921875 14
      Nvidia’s $1 Billion Bet Turns Nokia Into The New AI Contender
T+1D :
  min: 0.2711391150951385 732
       Cray Claims The Fastest Supercomputing Crown With The NVIDIA-Powered Titan
  max: 207.01727294921875 19
      Nvidia GTC-DC Event Highlights AI, Qua

In [288]:
# Extra EDA - check for predicted -100% drops in price... Stock value predicted to drop to zero = likely a missing value
df_sorted = df_combined.sort_values(by='qwasr_T+3D')
df_sorted.head(5)

,Unnamed: 0,Link,Title,Time,Author,Body,T-3D,T-1D,T-0D,T+1D,T+3D,relevancy,sentiment,qwasr_T+1D,qwasr_T+3D
228,228,https://www.forbes.com/sites/greatspeculations...,How Nvidia Stock Could Fall To $65,"Dec 20, 2024, 08:19am EST",Trefis Team,Could Nvidia stock fall by about 50% to levels...,130.347290,130.637161,134.655869,139.624237,139.884171,0.00,0.0,0.000,0.000
262,262,https://www.forbes.com/sites/daveywinder/2024/...,Urgent New Nvidia Security Warning For 200 Mil...,"Oct 25, 2024, 06:01am EDT",Davey Winder,With more than 200 million gamers using Nvidia...,143.533096,140.354340,141.483887,140.464294,139.284760,0.00,0.0,0.000,0.000
704,704,https://www.forbes.com/sites/antonyleather/201...,Nvidia Says VR Is Too Demanding For Most PCs: ...,"Jan 04, 2016, 08:23am EST",Antony Leather,Virtual Reality is touted as the next big thin...,0.821498,0.803936,0.789545,0.802229,0.738567,1.00,0.5,0.280,0.280
706,706,https://www.forbes.com/sites/marcochiappetta/2...,Microsoft Delivers Graphics-Intensive Applicat...,"Sep 29, 2015, 03:55pm EDT",Marco Chiappetta,NVIDIA just announced that Microsoft will offe...,0.569568,0.566166,0.576129,0.598970,0.602615,0.30,0.5,0.304,0.306
707,707,https://www.forbes.com/sites/jasonevangelho/20...,Nvidia Increases Desktop GPU Market Share Agai...,"Aug 19, 2015, 12:11pm EDT",Jason Evangelho,Despite the release of AMD's undeniably attrac...,0.569602,0.560821,0.558634,0.538466,0.503232,0.85,0.5,0.331,0.332


In [289]:
# Convert stock data into percent changes based around T-0D value
delta_df = pd.DataFrame()
label_delta_df = pd.DataFrame()

# True percent deltas from before article release
delta_df['d%_T-3D'] = (fyf_df['T-3D'] / fyf_df['T-0D']) - 1
delta_df['d%_T-1D'] = (fyf_df['T-1D'] / fyf_df['T-0D']) - 1

# QwASR predicted percent deltas after release
delta_df['qwasr_d%_T+1D'] = (q_df['qwasr_T+1D'] / fyf_df['T-0D']) - 1
delta_df['qwasr_d%_T+3D'] = (q_df['qwasr_T+3D'] / fyf_df['T-0D']) - 1

# True percent deltas after release
label_delta_df['true_d%_T+1D'] = (fyf_df['T+1D'] / fyf_df['T-0D']) - 1
label_delta_df['true_d%_T+3D'] = (fyf_df['T+3D'] / fyf_df['T-0D']) - 1

df_processed = pd.concat([q_df['relevancy'], q_df['sentiment'], fyf_df['T-0D'], delta_df, label_delta_df], axis=1)

df_processed.head(10)

,relevancy,sentiment,T-0D,d%_T-3D,d%_T-1D,qwasr_d%_T+1D,qwasr_d%_T+3D,true_d%_T+1D,true_d%_T+3D
0,0.90,-0.50,188.129349,0.056019,-0.000372,0.000003,0.000003,0.057933,0.030029
1,0.70,0.80,188.129349,0.056019,-0.000372,0.081596,0.086965,0.057933,0.030029
2,0.95,0.50,188.059357,0.099957,0.037909,-0.149205,-0.143887,0.000372,0.027010
3,1.00,1.00,195.188583,0.037293,0.017827,-0.187965,-0.180280,-0.036525,0.019671
4,0.70,-0.50,198.668182,0.021138,0.041220,-0.217288,-0.204704,-0.017515,-0.053047
5,0.95,0.95,206.857285,0.000773,-0.021220,-0.233771,-0.226520,-0.039588,-0.090874
6,1.00,1.00,206.857285,0.000773,-0.021220,-0.243005,-0.220235,-0.039588,-0.090874
7,0.85,0.50,202.467773,-0.007210,0.001975,-0.266649,-0.263438,0.021680,-0.035952
8,0.90,0.50,202.467773,-0.007210,0.001975,-0.285269,-0.281367,0.021680,-0.035952
9,0.95,0.70,202.467773,-0.007210,0.001975,-0.280034,-0.272180,0.021680,-0.035952


In [290]:
# Extra EDA - print min and max of each column, with index and article title
for col_name in df_processed:
  print(col_name, ":")
  print("  min:", df_processed[col_name].min(), df_processed[col_name].idxmin())
  # print("     ", df_combined.at[df_combined[col_name].idxmin(), 'Title'])
  print("  max:", df_processed[col_name].max(), df_processed[col_name].idxmax())
  # print("     ", df_combined.at[df_combined[col_name].idxmax(), 'Title'])

relevancy :
  min: 0.0 142
  max: 1.0 3
sentiment :
  min: -1.0 103
  max: 1.0 3
T-0D :
  min: 0.2732018828392029 732
  max: 207.01727294921875 14
d%_T-3D :
  min: -0.2386627879582609 446
  max: 0.241935564950309 190
d%_T-1D :
  min: -0.1959452211894478 450
  max: 0.20435741085822268 190
qwasr_d%_T+1D :
  min: -1.0 228
  max: 604.705928086139 732
qwasr_d%_T+3D :
  min: -1.0 228
  max: 607.0851210596463 732
true_d%_T+1D :
  min: -0.16968169831960267 193
  max: 0.24369635795127276 451
true_d%_T+3D :
  min: -0.19480524737206395 196
  max: 0.3134784221543825 451


In [291]:
# Extra EDA - Print number of predictions that fall outside expected range
print("Predicted gains with more than 35% change:", (df_processed['qwasr_d%_T+3D'] > 0.35).sum())
print("Predicted losses with more than 35% change:", (df_processed['qwasr_d%_T+3D'] < -0.35).sum())

Predicted gains with more than 35% change: 134
Predicted losses with more than 35% change: 282


In [292]:
# Stock value predicted to drop to zero = likely a missing value
df_processed.loc[df_processed['qwasr_d%_T+1D'] == -1.0, 'qwasr_d%_T+1D'] = 0
df_processed.loc[df_processed['qwasr_d%_T+3D'] == -1.0, 'qwasr_d%_T+3D'] = 0
df_processed.sort_values(by='qwasr_d%_T+3D').head()

,relevancy,sentiment,T-0D,d%_T-3D,d%_T-1D,qwasr_d%_T+1D,qwasr_d%_T+3D,true_d%_T+1D,true_d%_T+3D
675,1.00,1.0,4.834426,0.004752,-0.010374,-0.880027,-0.877959,0.031530,0.056825
674,0.85,0.6,5.035766,-0.049941,-0.009713,-0.864966,-0.862980,0.014571,0.010303
676,0.95,0.5,4.156582,0.022030,-0.073872,-0.865274,-0.862868,-0.008432,-0.041152
679,0.30,-0.2,3.428438,-0.081457,-0.020878,-0.857661,-0.859411,-0.013463,-0.004608
677,0.90,0.7,4.156582,0.022030,-0.073872,-0.859259,-0.855651,-0.008432,-0.041152


In [293]:
# Data normalization

df_final = pd.DataFrame()

df_final['relevancy'] = (df_processed['relevancy'] - 0.5) * 2 # [0,1] -> [-1, 1]
df_final['sentiment'] = df_processed['sentiment'] # Already normalized

# Scale relative to max price, than convert range [0, 1] -> [-1, 1]
df_final['price_T-0D'] = ((df_processed['T-0D'] / df_processed['T-0D'].max()) - 0.5) * 2

# Peacewise normalization
# If value is positive relative to most positive value
# If negative, scale relative to most negative value
cols = ['d%_T-3D', 'd%_T-1D', 'qwasr_d%_T+1D',
        'qwasr_d%_T+3D', 'true_d%_T+1D', 'true_d%_T+3D']

for col in cols:
  x = df_processed[col]

  max_val = x.max()
  min_val = x.min()

  # Avoid division by zero
  pos_scale = max_val if max_val != 0 else 1
  neg_scale = abs(min_val) if min_val != 0 else 1

  df_final[col] = np.where(
    x >= 0,
    x / pos_scale,
    x / neg_scale
  )

df_final.head(10)

,relevancy,sentiment,price_T-0D,d%_T-3D,d%_T-1D,qwasr_d%_T+1D,qwasr_d%_T+3D,true_d%_T+1D,true_d%_T+3D
0,0.8,-0.50,0.817523,0.231545,-0.001899,5.724581e-09,5.702146e-09,0.237724,0.095794
1,0.4,0.80,0.817523,0.231545,-0.001899,1.349354e-04,1.432499e-04,0.237724,0.095794
2,0.9,0.50,0.816847,0.413157,0.185506,-1.695457e-01,-1.638885e-01,0.001527,0.086161
3,1.0,1.00,0.885723,0.154145,0.087234,-2.135898e-01,-2.053399e-01,-0.215255,0.062751
4,0.4,-0.50,0.919339,0.087372,0.201705,-2.469104e-01,-2.331591e-01,-0.103220,-0.272310
5,0.9,0.95,0.998454,0.003197,-0.108296,-2.656410e-01,-2.580075e-01,-0.233308,-0.466486
6,1.0,1.00,0.998454,0.003197,-0.108296,-2.761332e-01,-2.508493e-01,-0.233308,-0.466486
7,0.7,0.50,0.956047,-0.030211,0.009666,-3.030006e-01,-3.000578e-01,0.088963,-0.184555
8,0.8,0.50,0.956047,-0.030211,0.009666,-3.241593e-01,-3.204788e-01,0.088963,-0.184555
9,0.9,0.70,0.956047,-0.030211,0.009666,-3.182102e-01,-3.100151e-01,0.088963,-0.184555


In [294]:
# Verify range of data - all values should be very close to [-1, 1]
for col_name in df_final:
  print(col_name, ":")
  print("  min:", df_final[col_name].min(), df_final[col_name].idxmin())
  # print("     ", df_combined.at[df_combined[col_name].idxmin(), 'Title'])
  print("  max:", df_final[col_name].max(), df_final[col_name].idxmax())
  # print("     ", df_combined.at[df_combined[col_name].idxmax(), 'Title'])

relevancy :
  min: -1.0 142
  max: 1.0 3
sentiment :
  min: -1.0 103
  max: 1.0 3
price_T-0D :
  min: -0.9973605885253235 732
  max: 1.0 14
d%_T-3D :
  min: -1.0 446
  max: 1.0 190
d%_T-1D :
  min: -1.0 450
  max: 1.0 190
qwasr_d%_T+1D :
  min: -1.0 675
  max: 1.0 732
qwasr_d%_T+3D :
  min: -1.0 675
  max: 1.0 732
true_d%_T+1D :
  min: -1.0 193
  max: 1.0 451
true_d%_T+3D :
  min: -1.0 196
  max: 1.0 451


In [295]:
# Export to csv
df_final.to_csv('all_data_preprocessed.csv', index=True)
df_final.head(800)

,relevancy,sentiment,price_T-0D,d%_T-3D,d%_T-1D,qwasr_d%_T+1D,qwasr_d%_T+3D,true_d%_T+1D,true_d%_T+3D
0,0.8,-0.5,0.817523,0.231545,-0.001899,5.724581e-09,5.702146e-09,0.237724,0.095794
1,0.4,0.8,0.817523,0.231545,-0.001899,1.349354e-04,1.432499e-04,0.237724,0.095794
2,0.9,0.5,0.816847,0.413157,0.185506,-1.695457e-01,-1.638885e-01,0.001527,0.086161
3,1.0,1.0,0.885723,0.154145,0.087234,-2.135898e-01,-2.053399e-01,-0.215255,0.062751
4,0.4,-0.5,0.919339,0.087372,0.201705,-2.469104e-01,-2.331591e-01,-0.103220,-0.272310
...,...,...,...,...,...,...,...,...,...
733,1.0,0.5,-0.997312,0.173638,0.060462,9.786981e-01,9.798945e-01,0.101403,0.068319
734,1.0,1.0,-0.997071,-0.044340,-0.088725,8.908932e-01,8.918562e-01,-0.169275,-0.376367
735,1.0,1.0,-0.996803,0.057248,0.006778,8.283441e-01,8.454043e-01,0.031259,0.083947
736,1.0,1.0,-0.996497,0.164601,0.034024,7.485993e-01,7.592944e-01,0.023345,-0.113570
